# 🔄 Module 5.4: Transfer Learning

## Standing on the Shoulders of Giants — Pre-trained Models for Vision & RL

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_05_Deep_Learning_For_Images/04_Transfer_Learning/transfer_learning.ipynb)

---

## 🎯 Learning Objectives

1. Understand why transfer learning works mathematically
2. Compare feature extraction vs fine-tuning approaches
3. Load and use pre-trained models (ResNet, VGG)
4. Visualize features from pre-trained models
5. Apply pre-trained features as RL state representations

---

In [ ]:
# Google Colab Setup
!pip install torch torchvision numpy matplotlib scikit-learn -q

In [ ]:
# Download REAL open-source datasets (TINY - under 250MB total)
import torchvision
import torchvision.transforms as transforms

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

# MNIST (handwritten digits - 11MB)
mnist_train = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
mnist_test = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# CIFAR-10 (real photographs - 170MB)
transform_cifar = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))])
cifar_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_cifar)
cifar_test = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_cifar)

# FashionMNIST (clothing items - 30MB, great for transfer learning!)
fashion_train = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
fashion_test = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

print(f"✅ MNIST: {len(mnist_train)} train + {len(mnist_test)} test (28x28 grayscale)")
print(f"✅ CIFAR-10: {len(cifar_train)} train + {len(cifar_test)} test (32x32 RGB)")
print(f"✅ FashionMNIST: {len(fashion_train)} train + {len(fashion_test)} test (28x28)")
print(f"   Classes: {cifar_train.classes}")
print(f"\n📦 Total download: ~211MB (NO STL-10 needed!)")

## Deep Derivation: Transfer Learning Theory and Fine-Tuning Mathematics

### Step 1: Domain Adaptation Bound (Ben-David et al. 2010)
Let $\mathcal{D}_S$ (source) and $\mathcal{D}_T$ (target) be two distributions. For hypothesis $h$:
$$\epsilon_T(h) \leq \epsilon_S(h) + \frac{1}{2}d_{\mathcal{H}\Delta\mathcal{H}}(\mathcal{D}_S, \mathcal{D}_T) + \lambda^*$$

where:
- $\epsilon_T(h)$: target error
- $\epsilon_S(h)$: source error  
- $d_{\mathcal{H}\Delta\mathcal{H}}$: $\mathcal{H}$-divergence between domains
- $\lambda^* = \min_h [\epsilon_S(h) + \epsilon_T(h)]$: ideal joint error

**Interpretation:** Transfer works when (1) source error is low, (2) domains are similar, and (3) a good joint hypothesis exists.

### Step 2: Fine-Tuning as Regularized Optimization
Fine-tuning with pretrained weights $\theta_0$:
$$\min_\theta \mathcal{L}_{\text{target}}(\theta) + \frac{\lambda}{2}\|\theta - \theta_0\|^2$$

**Proof this keeps parameters near pretrained values:**

Taking gradient: $\nabla_\theta = \nabla \mathcal{L}_{\text{target}} + \lambda(\theta - \theta_0)$

At equilibrium: $\nabla \mathcal{L}_{\text{target}}(\theta^*) = -\lambda(\theta^* - \theta_0)$

So $\|\theta^* - \theta_0\| = \frac{1}{\lambda}\|\nabla \mathcal{L}_{\text{target}}(\theta^*)\|$ — larger $\lambda$ keeps weights closer to $\theta_0$. $\blacksquare$

### Step 3: Layer-wise Learning Rate Decay
For layer $l$ out of $L$ total layers:
$$\eta_l = \eta_{\text{base}} \cdot \xi^{L-l}, \quad 0 < \xi < 1$$

**Derivation:** Early layers learn general features (edges, textures) that transfer well — small updates preserve them. Later layers are task-specific — larger updates adapt them.

**Proof learning rate decay preserves early features:**

After $T$ steps, weight change in layer $l$: $\|\Delta W^{(l)}\| \leq T \cdot \eta_l \cdot G$ where $G$ is max gradient norm.

Ratio: $\frac{\|\Delta W^{(1)}\|}{\|\Delta W^{(L)}\|} = \xi^{L-1} \ll 1$ for $\xi = 0.1, L = 50$: ratio $= 10^{-49}$. $\blacksquare$

### Step 4: Feature Reuse — CKA Similarity
Centered Kernel Alignment measures feature similarity between layers:
$$\text{CKA}(\mathbf{X}, \mathbf{Y}) = \frac{\|\mathbf{Y}^T\mathbf{X}\|_F^2}{\|\mathbf{X}^T\mathbf{X}\|_F \cdot \|\mathbf{Y}^T\mathbf{Y}\|_F}$$

**Proof CKA is invariant to isotropic scaling:**

Let $\mathbf{X}' = \alpha\mathbf{X}$. Then:
$$\text{CKA}(\alpha\mathbf{X}, \mathbf{Y}) = \frac{\|\mathbf{Y}^T(\alpha\mathbf{X})\|_F^2}{\|(\alpha\mathbf{X})^T(\alpha\mathbf{X})\|_F \cdot \|\mathbf{Y}^T\mathbf{Y}\|_F} = \frac{\alpha^2\|\mathbf{Y}^T\mathbf{X}\|_F^2}{\alpha^2\|\mathbf{X}^T\mathbf{X}\|_F \cdot \|\mathbf{Y}^T\mathbf{Y}\|_F} = \text{CKA}(\mathbf{X}, \mathbf{Y}) \quad \blacksquare$$

### Step 5: When NOT to Transfer — Negative Transfer
Transfer hurts when source and target tasks are dissimilar:
$$\epsilon_T(h_{\text{transfer}}) > \epsilon_T(h_{\text{scratch}})$$

**Condition for negative transfer:** $d_{\mathcal{H}\Delta\mathcal{H}}(\mathcal{D}_S, \mathcal{D}_T) > 2[\epsilon_T(h_{\text{scratch}}) - \epsilon_S(h_{\text{transfer}})]$

### Step 6: Number of Trainable Parameters
For a pretrained model with $P$ total parameters and $L$ layers:
- **Feature extraction** (freeze all but last): trainable $= C_{\text{out}} \times d_{\text{features}} + C_{\text{out}}$
- **Fine-tune last $k$ layers**: trainable $= \sum_{l=L-k+1}^{L} |W^{(l)}|$
- **Full fine-tune**: trainable $= P$

### Step 7: Connection to RL — Pre-trained Vision Backbone
In vision-based RL, a pretrained CNN (e.g., ResNet on ImageNet) serves as the state encoder:
$$s_t = f_{\theta_{\text{frozen}}}(\text{image}_t)$$

The RL policy $\pi_\phi(a|s)$ is trained on these fixed features. This reduces sample complexity from $O(|\mathcal{S}_{\text{pixel}}|)$ to $O(|\mathcal{S}_{\text{feature}}|)$ — a massive reduction since $d_{\text{feature}} \ll H \times W \times C$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset
from torchvision import models, transforms, datasets
import torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import time
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print('All libraries loaded!')

---

## Part 1: Why Transfer Learning Works

### 1.1 The Theoretical Foundation

A model trained on task $\mathcal{T}_A$ with data distribution $\mathcal{D}_A$ learns a feature mapping:

$$\phi_A: \mathcal{X} \rightarrow \mathbb{R}^d$$

**Transfer learning** re-uses $\phi_A$ for a new task $\mathcal{T}_B$ with distribution $\mathcal{D}_B$.

This works when the **feature transfer gap** is small:

$$\mathcal{R}_{\mathcal{T}_B}(\phi_A) \leq \mathcal{R}_{\mathcal{T}_B}(\phi_B^*) + \underbrace{d_{\mathcal{H}}(\mathcal{D}_A, \mathcal{D}_B)}_{\text{domain distance}} + \underbrace{\lambda^*}_{\text{task similarity}}$$

where $\mathcal{R}$ is the risk (error), and $d_{\mathcal{H}}$ measures distribution divergence.

### 1.2 Why ImageNet Features Transfer Well

Models trained on ImageNet (1.2M images, 1000 classes) learn **universal visual features**:

| Layer Depth | Features Learned | Transferability |
|-------------|-----------------|----------------|
| Conv1-2 | Gabor filters, color blobs | Very high (almost identical across tasks) |
| Conv3-4 | Textures, patterns | High |
| Conv5+ | Object parts, compositions | Task-dependent |
| FC layers | Task-specific combinations | Low (usually replaced) |

### 1.3 Two Transfer Strategies

**Strategy 1: Feature Extraction** — Freeze all CNN layers, train only the new classifier:
$$\theta^* = \arg\min_\theta \mathcal{L}(f_\theta(\phi_{\text{frozen}}(x)), y)$$

**Strategy 2: Fine-Tuning** — Unfreeze some/all layers and train with a small learning rate:
$$\theta^*, \phi^* = \arg\min_{\theta, \phi} \mathcal{L}(f_\theta(\phi(x)), y) \quad \text{with } \phi \text{ initialized from } \phi_A$$

In [ ]:
# Compare architectures side by side
def show_model_comparison():
    """Display architecture comparison of popular pre-trained models."""
    model_info = {
        'VGG16': {'params': 138_357_544, 'top1': 71.59, 'feat_dim': 4096, 'year': 2014},
        'ResNet18': {'params': 11_689_512, 'top1': 69.76, 'feat_dim': 512, 'year': 2015},
        'ResNet50': {'params': 25_557_032, 'top1': 76.13, 'feat_dim': 2048, 'year': 2015},
        'ResNet101': {'params': 44_549_160, 'top1': 77.37, 'feat_dim': 2048, 'year': 2015},
        'MobileNetV2': {'params': 3_504_872, 'top1': 71.88, 'feat_dim': 1280, 'year': 2018},
        'EfficientNet-B0': {'params': 5_288_548, 'top1': 77.69, 'feat_dim': 1280, 'year': 2019},
    }

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    names = list(model_info.keys())
    params = [model_info[n]['params'] / 1e6 for n in names]
    acc = [model_info[n]['top1'] for n in names]
    feat_dims = [model_info[n]['feat_dim'] for n in names]

    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(names)))

    axes[0].barh(names, params, color=colors, edgecolor='k')
    axes[0].set_xlabel('Parameters (millions)')
    axes[0].set_title('Model Size')
    for i, v in enumerate(params):
        axes[0].text(v + 1, i, f'{v:.1f}M', va='center', fontsize=9)

    axes[1].barh(names, acc, color=colors, edgecolor='k')
    axes[1].set_xlabel('Top-1 Accuracy (%)')
    axes[1].set_title('ImageNet Accuracy')
    axes[1].set_xlim(65, 82)
    for i, v in enumerate(acc):
        axes[1].text(v + 0.2, i, f'{v:.1f}%', va='center', fontsize=9)

    axes[2].barh(names, feat_dims, color=colors, edgecolor='k')
    axes[2].set_xlabel('Feature Vector Dimension')
    axes[2].set_title('Feature Dim (for RL state)')
    for i, v in enumerate(feat_dims):
        axes[2].text(v + 20, i, str(v), va='center', fontsize=9)

    plt.suptitle('Pre-trained Model Comparison', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()

show_model_comparison()

## Deep Dive: PAC-Bayes Transfer Bounds and Information-Theoretic View

### PAC-Bayes Bound for Transfer Learning

The PAC-Bayes framework provides a principled way to understand why pre-trained models generalize well after fine-tuning on a new task.

**Classical PAC-Bayes Bound:** For a data-independent prior $P$ over hypotheses and a data-dependent posterior $Q$, with probability $\geq 1 - \delta$ over the draw of $m$ training samples:

$$\mathbb{E}_{h \sim Q}[\text{err}(h)] \leq \mathbb{E}_{h \sim Q}[\hat{\text{err}}(h)] + \sqrt{\frac{D_{\text{KL}}(Q \| P) + \ln(2m/\delta)}{2m}}$$

### Application to Transfer Learning

**Step 1 — Prior as pre-trained model:** Set the prior $P = \mathcal{N}(\theta_{\text{pre}}, \sigma_P^2 \mathbf{I})$ centered on the pre-trained weights $\theta_{\text{pre}}$.

**Step 2 — Posterior as fine-tuned model:** Set the posterior $Q = \mathcal{N}(\theta_{\text{fine}}, \sigma_Q^2 \mathbf{I})$ centered on the fine-tuned weights $\theta_{\text{fine}}$.

**Step 3 — Compute the KL divergence:**

$$D_{\text{KL}}(Q \| P) = \frac{1}{2}\left[\frac{d \cdot \sigma_Q^2}{\sigma_P^2} + \frac{\|\theta_{\text{fine}} - \theta_{\text{pre}}\|^2}{\sigma_P^2} - d + d \ln\frac{\sigma_P^2}{\sigma_Q^2}\right]$$

where $d$ is the number of parameters.

**Key insight:** The KL penalty $\frac{\|\theta_{\text{fine}} - \theta_{\text{pre}}\|^2}{\sigma_P^2}$ is **small** when the fine-tuned weights stay close to the pre-trained weights. This explains why:

1. **Fine-tuning with weight decay** ($L_2$ regularization toward $\theta_{\text{pre}}$) has a tighter generalization bound
2. **Freezing early layers** reduces the effective number of parameters that change, shrinking $\|\theta_{\text{fine}} - \theta_{\text{pre}}\|^2$
3. **Small learning rates** during fine-tuning keep $\theta_{\text{fine}}$ near $\theta_{\text{pre}}$

### Information-Theoretic Interpretation

The KL term $D_{\text{KL}}(Q \| P)$ measures the **information gained** from the target data beyond what the pre-trained model already knows. Transfer learning works when:

$$\underbrace{D_{\text{KL}}(Q \| P)}_{\text{info gained from target data}} \ll \underbrace{D_{\text{KL}}(Q \| Q_{\text{scratch}})}_{\text{info needed to learn from scratch}}$$

**Proof that transfer reduces sample complexity:** From the PAC-Bayes bound, the number of samples needed for $\epsilon$-accuracy scales as:

$$m_{\text{transfer}} \sim \frac{D_{\text{KL}}(Q \| P_{\text{pre}})}{\epsilon^2}, \qquad m_{\text{scratch}} \sim \frac{D_{\text{KL}}(Q \| P_{\text{uniform}})}{\epsilon^2}$$

Since $D_{\text{KL}}(Q \| P_{\text{pre}}) \ll D_{\text{KL}}(Q \| P_{\text{uniform}})$ for related tasks, $m_{\text{transfer}} \ll m_{\text{scratch}}$. $\blacksquare$

### Practical Implication for Vision

For ImageNet pre-trained models fine-tuned on a target vision task with $m$ samples, the generalization error is approximately:

$$\text{err}_{\text{test}} \lesssim \text{err}_{\text{train}} + O\left(\sqrt{\frac{k \cdot \|\Delta\theta\|^2}{m}}\right)$$

where $k$ is the number of fine-tuned parameters and $\|\Delta\theta\|$ is the weight displacement. Freezing more layers reduces both $k$ and $\|\Delta\theta\|$, explaining the empirical success of partial fine-tuning with limited data.

---

## Part 2: Loading Pre-trained Models

### 2.1 ResNet Architecture

ResNet introduced **skip connections** to solve the degradation problem:

$$\mathbf{y} = \mathcal{F}(\mathbf{x}, \{W_i\}) + \mathbf{x}$$

where $\mathcal{F}(\mathbf{x}, \{W_i\})$ is the residual mapping. The gradient flows directly through the skip connection:

$$\frac{\partial \mathcal{L}}{\partial \mathbf{x}} = \frac{\partial \mathcal{L}}{\partial \mathbf{y}} \left(1 + \frac{\partial \mathcal{F}}{\partial \mathbf{x}}\right)$$

The "$+1$" term ensures gradients never vanish, enabling training of 100+ layer networks.

### 2.2 VGG Architecture

VGG uses a simple stack of 3×3 convolutions. Two 3×3 convolutions have the same receptive field as one 5×5 but with fewer parameters:

$$\text{Parameters: } 2 \times (3^2 C^2) = 18C^2 \quad < \quad 5^2 C^2 = 25C^2$$

In [ ]:
# Load pre-trained models
resnet18 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet50 = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
vgg16 = models.vgg16(weights=models.VGG16_Weights.DEFAULT)

for name, model in [('ResNet18', resnet18), ('ResNet50', resnet50), ('VGG16', vgg16)]:
    total = sum(p.numel() for p in model.parameters())
    print(f'{name:10s}: {total:>12,} parameters')

# Show ResNet18 architecture breakdown
print('\n--- ResNet18 Layer Details ---')
resnet18.eval()
x_dummy = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    x = resnet18.conv1(x_dummy)
    print(f'conv1:   {x.shape}')
    x = resnet18.bn1(x)
    x = resnet18.relu(x)
    x = resnet18.maxpool(x)
    print(f'maxpool: {x.shape}')
    x = resnet18.layer1(x)
    print(f'layer1:  {x.shape}')
    x = resnet18.layer2(x)
    print(f'layer2:  {x.shape}')
    x = resnet18.layer3(x)
    print(f'layer3:  {x.shape}')
    x = resnet18.layer4(x)
    print(f'layer4:  {x.shape}')
    x = resnet18.avgpool(x)
    print(f'avgpool: {x.shape}  ← This is the feature vector!')
    x = torch.flatten(x, 1)
    print(f'flatten: {x.shape}')

---

## Part 3: Feature Extraction — Freeze and Classify

### 3.1 The Approach

1. Take a pre-trained CNN (e.g., ResNet18)
2. **Remove** the final classification layer
3. **Freeze** all convolutional weights (no gradient updates)
4. Pass images through to get feature vectors
5. Train a simple classifier (linear or small MLP) on these features

This is fast because we only train the small classifier head.

In [ ]:
# Create a small dataset using CIFAR-10
transform_cifar = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

cifar_train = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_cifar)
cifar_test = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_cifar)

class_names = cifar_train.classes
print(f'CIFAR-10 classes: {class_names}')

# Use a small subset for demonstration
n_train_per_class = 100  # Only 100 images per class!
n_test_per_class = 50

def get_subset(dataset, n_per_class):
    indices = []
    counts = {i: 0 for i in range(10)}
    for idx, (_, label) in enumerate(dataset):
        if counts[label] < n_per_class:
            indices.append(idx)
            counts[label] += 1
        if all(c >= n_per_class for c in counts.values()):
            break
    return Subset(dataset, indices)

train_subset = get_subset(cifar_train, n_train_per_class)
test_subset = get_subset(cifar_test, n_test_per_class)

print(f'Training samples: {len(train_subset)} ({n_train_per_class} per class)')
print(f'Test samples: {len(test_subset)} ({n_test_per_class} per class)')
print('\nWith only 100 samples per class, transfer learning shines!')

In [ ]:
def extract_features_batch(model, dataset, device, batch_size=32):
    """Extract features from a pre-trained model for the entire dataset."""
    # Remove the final FC layer
    if isinstance(model, models.ResNet):
        feature_extractor = nn.Sequential(*list(model.children())[:-1])  # Remove FC
    elif isinstance(model, models.VGG):
        feature_extractor = model.features
    else:
        raise ValueError('Unsupported model type')

    feature_extractor = feature_extractor.to(device)
    feature_extractor.eval()

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    all_features = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            features = feature_extractor(images)
            features = features.view(features.size(0), -1)  # Flatten

            if isinstance(model, models.VGG):
                # VGG features need adaptive avg pool
                features_pooled = nn.AdaptiveAvgPool2d(1)(
                    feature_extractor(images)
                ).view(images.size(0), -1)
                features = features_pooled

            all_features.append(features.cpu().numpy())
            all_labels.append(labels.numpy())

    return np.concatenate(all_features), np.concatenate(all_labels)


print('Extracting features with ResNet18...')
t0 = time.time()
train_features_r18, train_labels = extract_features_batch(resnet18, train_subset, device)
test_features_r18, test_labels = extract_features_batch(resnet18, test_subset, device)
t_r18 = time.time() - t0

print(f'  Done in {t_r18:.1f}s')
print(f'  Train features: {train_features_r18.shape}')
print(f'  Test features: {test_features_r18.shape}')

print('\nExtracting features with ResNet50...')
t0 = time.time()
train_features_r50, _ = extract_features_batch(resnet50, train_subset, device)
test_features_r50, _ = extract_features_batch(resnet50, test_subset, device)
t_r50 = time.time() - t0
print(f'  Done in {t_r50:.1f}s')
print(f'  Features shape: {train_features_r50.shape}')

In [ ]:
# Train simple classifiers on extracted features
results = {}

for name, train_feat, test_feat in [
    ('ResNet18', train_features_r18, test_features_r18),
    ('ResNet50', train_features_r50, test_features_r50),
]:
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(train_feat, train_labels)
    train_acc = accuracy_score(train_labels, clf.predict(train_feat))
    test_acc = accuracy_score(test_labels, clf.predict(test_feat))
    results[name] = {'train': train_acc, 'test': test_acc}
    print(f'{name} + LogisticRegression:')
    print(f'  Train accuracy: {train_acc:.2%}')
    print(f'  Test accuracy:  {test_acc:.2%}\n')

# Compare
fig, ax = plt.subplots(figsize=(10, 5))
x_pos = np.arange(len(results))
width = 0.35
train_accs = [results[n]['train'] for n in results]
test_accs = [results[n]['test'] for n in results]

bars1 = ax.bar(x_pos - width/2, train_accs, width, label='Train', color='steelblue', edgecolor='k')
bars2 = ax.bar(x_pos + width/2, test_accs, width, label='Test', color='coral', edgecolor='k')

ax.set_ylabel('Accuracy')
ax.set_title('Feature Extraction: Pre-trained Features + Linear Classifier\n'
             f'(Only {n_train_per_class} training samples per class!)', fontsize=13)
ax.set_xticks(x_pos)
ax.set_xticklabels(results.keys())
ax.legend()
ax.set_ylim(0, 1.05)

for bar, acc in zip(list(bars1) + list(bars2), train_accs + test_accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{acc:.1%}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print('Key insight: With only a linear classifier on frozen features,')
print('pre-trained ResNet achieves strong accuracy even with very few samples!')

## Deep Dive: Feature Transferability — Which Layers to Freeze

### Empirical Observation: The Transferability Gap

Yosinski et al. (2014) showed that CNN features transition from **general** (early layers) to **task-specific** (later layers). The **transferability** of layer $l$ from task $A$ to task $B$ can be quantified as:

$$\mathcal{T}(l) = \text{Acc}_{B}(\text{freeze layers } 1\ldots l \text{ from } A) - \text{Acc}_{B}(\text{train from scratch})$$

### Theoretical Justification: Feature Complexity Hierarchy

**Step 1 — Gabor-like filters in layer 1:** The first convolutional layer learns oriented edge detectors that approximate Gabor filters:

$$g(x, y; \theta, \lambda, \sigma) = \exp\left(-\frac{x'^2 + \gamma^2 y'^2}{2\sigma^2}\right) \cos\left(\frac{2\pi x'}{\lambda}\right)$$

where $x' = x\cos\theta + y\sin\theta$, $y' = -x\sin\theta + y\cos\theta$.

These filters are **universal** — they are optimal for detecting edges regardless of the task (natural images, medical images, satellite imagery). This explains why layer 1 almost always transfers perfectly.

**Step 2 — Composition hierarchy:** Let $\phi_l$ denote the features at layer $l$. The layer-wise feature complexity follows a hierarchical pattern:

| Layer | Feature Type | Universality | Transfer $\mathcal{T}(l)$ |
|-------|-------------|-------------|---------------------------|
| 1 | Edges, color blobs | Very high | $\approx +5\%$ |
| 2 | Textures, corners | High | $\approx +4\%$ |
| 3 | Patterns, parts | Medium | $\approx +2\%$ |
| 4-5 | Object parts | Task-dependent | $\approx 0$ to $+1\%$ |
| FC | Class-specific | Low | Often negative |

**Step 3 — The co-adaptation problem:** When transferring intermediate layers, there is a **fragile co-adaptation** between consecutive layers. Layer $l+1$ is trained to expect specific outputs from layer $l$. Freezing only some layers can break these co-adaptations.

**Formal statement:** Let $f = f_L \circ f_{L-1} \circ \cdots \circ f_1$ be the network decomposition. Freezing layers $1, \ldots, k$ and re-initializing layers $k+1, \ldots, L$ creates a distribution mismatch:

$$p(\phi_{k+1}^{\text{new}} | \phi_k^{\text{old}}) \neq p(\phi_{k+1}^{\text{original}} | \phi_k^{\text{old}})$$

This mismatch is minimized when the frozen/trainable boundary is placed at a **natural feature hierarchy boundary** (e.g., between a pooling layer and the next convolution block).

### Optimal Freezing Strategy

**For similar domains** (e.g., ImageNet → CIFAR-10): Freeze all but the last 1-2 blocks + classifier.

**For dissimilar domains** (e.g., ImageNet → medical images): Freeze only layer 1 (edges transfer), fine-tune everything else with a small learning rate.

**Mathematical criterion:** Freeze layer $l$ if:

$$\frac{\|\nabla_{W_l} \mathcal{L}_{\text{target}}\|}{\|W_l^{\text{pre}}\|} < \tau$$

i.e., the relative gradient magnitude is below a threshold $\tau$, indicating the pre-trained features are already near-optimal for the target task.

---

## Part 4: Fine-Tuning — Adapting the Pre-trained Model

### 4.1 Fine-Tuning Strategy

Instead of training only a linear classifier, we can fine-tune some or all layers:

$$\theta_{\text{fine-tuned}} = \theta_{\text{pre-trained}} - \alpha_{\text{small}} \cdot \nabla_\theta \mathcal{L}_{\text{new task}}$$

**Critical**: Use a **small learning rate** to avoid destroying the pre-trained features!

### 4.2 Layer-wise Learning Rates

A common strategy: use different learning rates for different depths:

$$\alpha_l = \alpha_{\text{base}} \cdot \gamma^{L - l}$$

where $\gamma < 1$ (e.g., $\gamma = 0.1$), so early layers get smaller learning rates.

In [ ]:
class FineTunedResNet(nn.Module):
    """ResNet18 with a custom classification head for fine-tuning."""

    def __init__(self, num_classes=10, freeze_until='layer3'):
        super().__init__()
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        # Freeze early layers
        freeze_layers = True
        for name, param in self.backbone.named_parameters():
            if freeze_until in name:
                freeze_layers = False
            param.requires_grad = not freeze_layers

        # Replace the classifier head
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.backbone(x)


def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# Compare freezing strategies
strategies = {
    'Freeze all (feature extract)': 'fc',
    'Freeze until layer3': 'layer3',
    'Freeze until layer2': 'layer2',
    'Freeze until layer1': 'layer1',
}

print(f'{"Strategy":35s} | {"Trainable Params":>18s} | {"% of Total":>10s}')
print('-' * 70)

total_params = sum(p.numel() for p in models.resnet18().parameters())
for desc, freeze in strategies.items():
    m = FineTunedResNet(freeze_until=freeze)
    trainable = count_trainable(m)
    pct = trainable / total_params * 100
    print(f'{desc:35s} | {trainable:>18,} | {pct:>9.1f}%')

In [ ]:
def train_model(model, train_loader, test_loader, epochs=15, lr=0.001, device='cpu'):
    """Train a model and return training history."""
    model = model.to(device)
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    criterion = nn.CrossEntropyLoss()

    history = {'train_loss': [], 'train_acc': [], 'test_acc': []}

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += images.size(0)

        train_loss = total_loss / total
        train_acc = correct / total

        # Test
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                correct += (outputs.argmax(1) == labels).sum().item()
                total += images.size(0)
        test_acc = correct / total

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)

        if (epoch + 1) % 5 == 0:
            print(f'  Epoch {epoch+1:2d}/{epochs} — Loss: {train_loss:.4f} — '
                  f'Train: {train_acc:.2%} — Test: {test_acc:.2%}')

    return history


# Create data loaders
train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=32, shuffle=False)

# Train with different fine-tuning strategies
all_histories = {}

for desc, freeze in [('Feature extraction only', 'fc'),
                      ('Fine-tune layer4+', 'layer4'),
                      ('Fine-tune layer3+', 'layer3')]:
    print(f'\n=== {desc} ===')
    model_ft = FineTunedResNet(freeze_until=freeze)
    trainable = count_trainable(model_ft)
    print(f'  Trainable params: {trainable:,}')

    t0 = time.time()
    history = train_model(model_ft, train_loader, test_loader, epochs=15, lr=0.001, device=device)
    elapsed = time.time() - t0
    print(f'  Training time: {elapsed:.1f}s')
    all_histories[desc] = history

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors = ['#e74c3c', '#3498db', '#2ecc71']
for i, (name, hist) in enumerate(all_histories.items()):
    axes[0].plot(hist['train_loss'], color=colors[i], linewidth=2, label=name)
    axes[1].plot(hist['test_acc'], color=colors[i], linewidth=2, label=name)

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training Loss')
axes[0].set_title('Training Loss Comparison')
axes[0].legend(fontsize=9)

axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Test Accuracy')
axes[1].set_title('Test Accuracy Comparison')
axes[1].legend(fontsize=9)
axes[1].set_ylim(0, 1.05)

plt.suptitle('Feature Extraction vs Fine-Tuning\n'
             f'(CIFAR-10, {n_train_per_class} samples/class)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nFinal test accuracies:')
for name, hist in all_histories.items():
    print(f'  {name:30s}: {hist["test_acc"][-1]:.2%}')

## Deep Dive: Fine-Tuning as Regularized Optimization

### The Explicit Regularization View

Fine-tuning with $L_2$ regularization toward the pre-trained weights $\theta_0$ solves:

$$\theta^* = \arg\min_\theta \; \mathcal{L}_{\text{target}}(\theta) + \frac{\lambda}{2}\|\theta - \theta_0\|^2$$

### Full Analysis of the Regularized Solution

**Step 1 — First-order optimality condition:**

$$\nabla_\theta \mathcal{L}_{\text{target}}(\theta^*) + \lambda(\theta^* - \theta_0) = \mathbf{0}$$

$$\theta^* = \theta_0 - \frac{1}{\lambda}\nabla_\theta \mathcal{L}_{\text{target}}(\theta^*)$$

**Step 2 — Taylor expansion around $\theta_0$:** For quadratic loss (or local quadratic approximation):

$$\mathcal{L}_{\text{target}}(\theta) \approx \mathcal{L}_{\text{target}}(\theta_0) + \mathbf{g}^T(\theta - \theta_0) + \frac{1}{2}(\theta - \theta_0)^T \mathbf{H} (\theta - \theta_0)$$

where $\mathbf{g} = \nabla_\theta \mathcal{L}|_{\theta_0}$ and $\mathbf{H} = \nabla^2_\theta \mathcal{L}|_{\theta_0}$.

**Step 3 — Solve the regularized quadratic:**

$$\nabla_\theta [\mathcal{L} + \frac{\lambda}{2}\|\theta - \theta_0\|^2] = \mathbf{g} + \mathbf{H}(\theta - \theta_0) + \lambda(\theta - \theta_0) = \mathbf{0}$$

$$\theta^* - \theta_0 = -(\mathbf{H} + \lambda \mathbf{I})^{-1}\mathbf{g}$$

**Step 4 — Eigendecomposition insight:** Let $\mathbf{H} = \mathbf{U}\text{diag}(h_i)\mathbf{U}^T$ be the eigendecomposition. In the eigenbasis:

$$[\theta^* - \theta_0]_i = -\frac{g_i}{h_i + \lambda}$$

**Interpretation:** The weight change along the $i$-th eigendirection of the Hessian is:
- **Suppressed** when $\lambda \gg h_i$ (regularization dominates): the pre-trained direction is preserved
- **Allowed** when $h_i \gg \lambda$ (gradient signal dominates): the weight adapts to the target task

This provides a **spectral view of fine-tuning**: directions with high curvature (important for the target task) are updated; directions with low curvature (unimportant) retain pre-trained values.

### Connection to Bayesian Transfer Learning

The regularized objective has a Bayesian interpretation:
- **Prior:** $p(\theta) = \mathcal{N}(\theta_0, \lambda^{-1}\mathbf{I})$ centered on pre-trained weights
- **Likelihood:** $p(\mathcal{D}_{\text{target}} | \theta) \propto \exp(-\mathcal{L}_{\text{target}}(\theta))$
- **MAP estimate:** $\theta^* = \arg\max_\theta \log p(\theta | \mathcal{D}_{\text{target}}) = \arg\min_\theta [\mathcal{L}_{\text{target}} + \frac{\lambda}{2}\|\theta - \theta_0\|^2]$

**Proof:** By Bayes' rule:
$$\log p(\theta | \mathcal{D}) = \log p(\mathcal{D}|\theta) + \log p(\theta) - \log p(\mathcal{D})$$
$$= -\mathcal{L}_{\text{target}}(\theta) - \frac{\lambda}{2}\|\theta - \theta_0\|^2 + \text{const} \quad \blacksquare$$

### Practical $\lambda$ Selection

- **Large $\lambda$** ($\lambda \gg 1$): Weights stay near $\theta_0$ → effectively feature extraction
- **Small $\lambda$** ($\lambda \to 0$): Full fine-tuning, risk of overfitting on small target datasets
- **Rule of thumb:** $\lambda \propto 1/m_{\text{target}}$ where $m_{\text{target}}$ is the number of target samples. More data → less regularization needed.

---

## Part 5: Visualizing Pre-trained Feature Spaces

### 5.1 Feature Embeddings Across Models

Let's compare how different pre-trained models organize images in their feature space:

In [ ]:
# t-SNE visualization of ResNet18 features on CIFAR-10
print('Computing t-SNE embedding of ResNet18 features...')
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
features_2d = tsne.fit_transform(test_features_r18)

fig, ax = plt.subplots(figsize=(12, 8))
scatter_colors = plt.cm.tab10(np.linspace(0, 1, 10))

for class_idx in range(10):
    mask = test_labels == class_idx
    ax.scatter(features_2d[mask, 0], features_2d[mask, 1],
               c=[scatter_colors[class_idx]], label=class_names[class_idx],
               s=60, alpha=0.7, edgecolors='k', linewidths=0.5)

ax.set_xlabel('t-SNE Component 1')
ax.set_ylabel('t-SNE Component 2')
ax.set_title('ResNet18 Feature Space: CIFAR-10 Test Images\n'
             '(Features from pre-trained model, no fine-tuning)',
             fontsize=14)
ax.legend(fontsize=9, ncol=2, loc='best', framealpha=0.8)
plt.tight_layout()
plt.show()

print('Even without any training on CIFAR-10, ResNet18 features')
print('create meaningful clusters — this is the power of transfer learning!')

---

## Part 6: Pre-trained Features as RL State

### 6.1 The RL Pipeline

In Deep RL with image observations:

$$\text{Raw pixels} \xrightarrow{\phi_{\text{pre-trained}}} \text{Feature vector} \xrightarrow{Q_{\theta}} \text{Q-values / Policy}$$

**Advantages over training from scratch:**
1. Faster convergence (features already meaningful)
2. Better sample efficiency (need fewer RL interactions)
3. More robust (pre-trained features generalize well)

### 6.2 Simulating an RL State Encoder

In [ ]:
class RLStateEncoder(nn.Module):
    """
    Pre-trained CNN as RL state encoder.
    Converts raw image observations to compact state vectors.
    """

    def __init__(self, backbone='resnet18', state_dim=128, freeze_backbone=True):
        super().__init__()
        if backbone == 'resnet18':
            self.cnn = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
            feat_dim = self.cnn.fc.in_features  # 512
            self.cnn.fc = nn.Identity()
        elif backbone == 'resnet50':
            self.cnn = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
            feat_dim = self.cnn.fc.in_features  # 2048
            self.cnn.fc = nn.Identity()

        if freeze_backbone:
            for param in self.cnn.parameters():
                param.requires_grad = False

        # Projection to RL state dimension
        self.projector = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Linear(256, state_dim),
        )

        self.state_dim = state_dim
        self.feat_dim = feat_dim

    def forward(self, observation):
        """Convert image observation to state vector."""
        with torch.no_grad():
            features = self.cnn(observation)
        state = self.projector(features)
        return state


encoder = RLStateEncoder(backbone='resnet18', state_dim=128).to(device)
encoder.eval()

# Simulate encoding game frames
dummy_frames = torch.randn(4, 3, 224, 224).to(device)
with torch.no_grad():
    states = encoder(dummy_frames)

print('RL State Encoder:')
print(f'  Input:  game frame {tuple(dummy_frames.shape[1:])} → {np.prod(dummy_frames.shape[1:]):,} values')
print(f'  Output: state vector ({states.shape[1]},) → {states.shape[1]} values')
print(f'  Compression ratio: {np.prod(dummy_frames.shape[1:]) / states.shape[1]:.0f}x')
print(f'\n  Backbone params (frozen): {sum(p.numel() for p in encoder.cnn.parameters()):,}')
print(f'  Projector params (trainable): {sum(p.numel() for p in encoder.projector.parameters() if p.requires_grad):,}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Show state vectors
for i in range(4):
    axes[0].plot(states[i].cpu().numpy(), alpha=0.7, label=f'Frame {i}')
axes[0].set_xlabel('State Dimension')
axes[0].set_ylabel('Value')
axes[0].set_title('State Vectors for Different Observations')
axes[0].legend(fontsize=9)

# Pipeline diagram
axes[1].axis('off')
pipeline_text = (
    'RL with Pre-trained State Encoder:\n\n'
    '  Image (3×224×224)\n'
    '       ↓\n'
    '  [ResNet18 - FROZEN]\n'
    '       ↓\n'
    '  Feature vector (512)\n'
    '       ↓\n'
    '  [Projector - TRAINABLE]\n'
    '       ↓\n'
    '  State vector (128)\n'
    '       ↓\n'
    '  [Q-Network / Policy]\n'
    '       ↓\n'
    '  Action'
)
axes[1].text(0.5, 0.5, pipeline_text, transform=axes[1].transAxes,
             fontsize=12, verticalalignment='center', horizontalalignment='center',
             fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
axes[1].set_title('RL Pipeline with Transfer Learning', fontsize=13)

plt.tight_layout()
plt.show()

## Deep Dive: Domain Confusion and Gradient Reversal Layer

### The Domain Adaptation Problem

When source and target domains differ significantly, standard transfer may fail. **Domain adaptation** aims to learn features that are:
1. **Discriminative** for the source task
2. **Domain-invariant** (cannot distinguish source from target)

### The Gradient Reversal Layer (GRL) — Ganin et al. (2016)

**Architecture:** Three components trained jointly:
- **Feature extractor** $G_f(\cdot; \theta_f)$: maps inputs to features
- **Label predictor** $G_y(\cdot; \theta_y)$: predicts class labels from features
- **Domain classifier** $G_d(\cdot; \theta_d)$: predicts source vs. target from features

**Objective:**

$$\min_{\theta_f, \theta_y} \max_{\theta_d} \; \mathcal{L}_y(G_y(G_f(x; \theta_f); \theta_y), y) - \lambda \mathcal{L}_d(G_d(G_f(x; \theta_f); \theta_d), d)$$

where $y$ is the class label and $d \in \{0, 1\}$ is the domain label.

### Mathematical Mechanism of the GRL

**Forward pass:** The GRL acts as the identity: $\text{GRL}(\mathbf{x}) = \mathbf{x}$

**Backward pass:** The GRL negates the gradient: $\frac{\partial \text{GRL}}{\partial \mathbf{x}} = -\lambda \mathbf{I}$

**Proof that this implements the minimax objective:**

During backpropagation, the feature extractor receives:

$$\frac{\partial \mathcal{L}_{\text{total}}}{\partial \theta_f} = \frac{\partial \mathcal{L}_y}{\partial \theta_f} - \lambda \frac{\partial \mathcal{L}_d}{\partial \theta_f}$$

The first term pushes features to be class-discriminative. The second term (negated by GRL) pushes features to **confuse** the domain classifier — making them domain-invariant. $\blacksquare$

### Connection to $\mathcal{H}$-Divergence

The domain classifier's accuracy relates to the $\mathcal{H}$-divergence from Ben-David's bound:

$$d_{\mathcal{H}}(S, T) = 2\left(1 - \min_{h \in \mathcal{H}} \text{err}_{\text{domain}}(h)\right)$$

When the domain classifier cannot distinguish domains ($\text{err}_{\text{domain}} \approx 0.5$), we have $d_{\mathcal{H}} \approx 0$, and the transfer bound becomes tight:

$$\epsilon_T(h) \leq \epsilon_S(h) + 0 + \lambda^* = \epsilon_S(h) + \lambda^*$$

### The GRL Training Dynamics

**Step 1 — Early training:** Domain classifier easily distinguishes source from target. Features are domain-specific. $\mathcal{L}_d$ is low.

**Step 2 — GRL kicks in:** Gradient reversal forces the feature extractor to produce domain-confusing features. $\mathcal{L}_d$ increases toward $\log 2$ (random chance).

**Step 3 — Equilibrium:** At the saddle point, features are maximally domain-invariant while remaining class-discriminative. The domain classifier accuracy $\approx 50\%$.

**Convergence condition:** The GRL schedule $\lambda_p = \frac{2}{1 + \exp(-10p)} - 1$ (where $p$ is training progress from 0 to 1) starts with $\lambda \approx 0$ and gradually increases to $\lambda \approx 1$:

$$\lambda_p: \quad 0 \xrightarrow{p=0.3} 0.5 \xrightarrow{p=0.5} 0.9 \xrightarrow{p=1.0} 1.0$$

This schedule prevents the GRL from destabilizing early training when the label predictor has not yet converged.

---

## 🔑 Key Takeaways

| Concept | Key Insight |
|---------|------------|
| **Transfer learning** | Re-use features learned on large datasets (ImageNet) |
| **Feature extraction** | Freeze backbone, train only classifier — fast, works with few samples |
| **Fine-tuning** | Unfreeze some layers, use small learning rate — better accuracy |
| **ResNet** | Skip connections: $y = \mathcal{F}(x) + x$ — trains very deep networks |
| **For RL** | Pre-trained CNN → frozen feature extractor → compact state for RL agent |

### When to Use Each Strategy

| Scenario | Data Size | Similarity to ImageNet | Strategy |
|----------|-----------|----------------------|----------|
| Small + similar | < 1K/class | High | Feature extraction |
| Small + different | < 1K/class | Low | Fine-tune all, small lr |
| Large + similar | > 10K/class | High | Fine-tune top layers |
| Large + different | > 10K/class | Low | Train from scratch (or fine-tune all) |

---

## ➡️ Next: Module 5.5 — Image Classification

We'll build a complete end-to-end image classification pipeline on CIFAR-10 — the same type of CNN backbone that powers Deep RL agents like DQN and A3C!